<a href="https://colab.research.google.com/github/EdenGebremedhin/AgriMargin_Kaggle_Capstone_Project/blob/main/AgriMargin_Kaggle_Capstone_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AgriMargin - AI Agents Capstone Submission
### Kaggle: *5-Day AI Agents — Intensive Vibe Coding Capstone Project*
**Track: Agents for Good**

**Team Members:**

              - Dawit Ayele Woldesenbet
              - Eden Habtetsion Gebremedhin

An AI decision-agent that helps smallholder farmers decide whether treating
a crop problem is economically worth it, combining pest/disease diagnosis,
climate-risk assessment, and market price into one recommendation, and
(with explicit human confirmation) acting on it.



## 1. Setup

In [109]:
# Install Google's Agent Development Kit
!pip install -q google-adk


In [110]:
import os
from getpass import getpass

# Get a free key at https://aistudio.google.com/apikey
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Please enter your Google API key: ")
print("API key set." if os.environ.get("GOOGLE_API_KEY") else "No key set yet.")

API key set.


## 2. Tools

Two kinds of functions below, and it matters which is which:

- **Real logic** — `compute_disease_risk()` and `compute_economics()` are
  genuine, deterministic calculations. This is where the actual
  "intelligence" of the economic-decision layer lives.
- **Stub / TODO** — `get_weather_forecast()` and `get_market_price()`
  return plausible-shaped fake data so the whole pipeline is runnable
  end-to-end right now. They are clearly labeled `STUB DATA` in their
  return values — replace them with real APIs before this touches a real
  farmer.


In [111]:
%%writefile tools.py
"""
tools.py — Function tools used by the AgriMargin agents.

Two kinds of functions live here, and it matters which is which:

1. REAL LOGIC — compute_disease_risk() and compute_economics() are genuine,
   deterministic calculations. They don't call any external API and will
   give you real numbers today. This is where the actual "intelligence"
   of the economic-decision layer lives.

2. STUB / TODO — get_weather_forecast() and get_market_price() return
   plausible-shaped fake data so the whole pipeline is runnable end to end
   right now. Before this goes anywhere near a real farmer, swap these for
   real calls (a weather API, a market-price API/MCP server). They are
   clearly marked below — do not mistake their output for real data.

send_dealer_order() is wrapped with require_confirmation=True where it's
registered as a tool (see agents.py) — the agent cannot execute it without
an explicit human "yes" being returned to the tool_confirmation flow. This
is the safety rail the capstone rubric wants under "security features".
"""

from datetime import datetime, timedelta
import random


# ---------------------------------------------------------------------------
# 1. WEATHER — STUB. Replace with a real weather API (e.g. OpenWeatherMap,
#    or a national meteorological service API) before production use.
# ---------------------------------------------------------------------------
def get_weather_forecast(location: str) -> dict:
    """Get a short-term weather forecast for a farm location.

    Args:
        location: Town/district name or lat,lon string.

    Returns:
        dict with avg_temp_c, avg_humidity_pct, rain_expected_mm over the
        next 5 days.
    """
    # STUB: deterministic-but-fake numbers seeded by location name so demo
    # runs are repeatable. Replace this whole function body with a real
    # API call. Do not treat this output as real weather.
    seed = sum(ord(c) for c in location)
    rnd = random.Random(seed)
    return {
        "location": location,
        "forecast_days": 5,
        "avg_temp_c": round(rnd.uniform(22, 34), 1),
        "avg_humidity_pct": round(rnd.uniform(45, 90), 1),
        "rain_expected_mm": round(rnd.uniform(0, 40), 1),
        "source": "STUB DATA — replace with real weather API",
    }


# ---------------------------------------------------------------------------
# 2. DISEASE RISK — REAL LOGIC. A simplified but genuine heuristic: fungal
#    and bacterial crop disease spread risk rises with humidity and warm
#    temperatures. This is a simplified version of logic used in real
#    agronomy decision-support tools (e.g. blight risk models), not a
#    black box. Tune the thresholds per-crop with an agronomist before
#    relying on it for real decisions.
# ---------------------------------------------------------------------------
def compute_disease_risk(avg_temp_c: float, avg_humidity_pct: float, crop: str) -> dict:
    """Compute a 0-100 disease-spread risk index from weather conditions.

    Args:
        avg_temp_c: Average forecast temperature in Celsius.
        avg_humidity_pct: Average forecast relative humidity in percent.
        crop: Crop name (e.g. "tomato", "potato", "maize").

    Returns:
        dict with risk_score (0-100), risk_level, and rationale.
    """
    # Fungal pathogens (early blight, late blight, powdery mildew, etc.)
    # generally thrive at humidity > 80% and moderate-warm temps 20-28C.
    humidity_component = max(0.0, (avg_humidity_pct - 50)) * 1.4
    if 18 <= avg_temp_c <= 28:
        temp_component = 30
    elif 28 < avg_temp_c <= 34:
        temp_component = 18
    else:
        temp_component = 8

    risk_score = min(100, round(humidity_component * 0.6 + temp_component))

    if risk_score >= 70:
        level = "high"
    elif risk_score >= 40:
        level = "moderate"
    else:
        level = "low"

    return {
        "crop": crop,
        "risk_score": risk_score,
        "risk_level": level,
        "rationale": (
            f"At {avg_humidity_pct}% humidity and {avg_temp_c}C, conditions are "
            f"{'favorable' if risk_score >= 40 else 'not strongly favorable'} "
            f"for fungal disease spread on {crop}."
        ),
    }


# ---------------------------------------------------------------------------
# 3. MARKET PRICES — STUB. Replace with a real mandi/market-price API
#    (e.g. a government open-data ag-market API for your region) or an
#    MCP server exposing that data. See README for how to swap this for
#    an MCPToolset connection.
# ---------------------------------------------------------------------------
def get_market_price(crop: str, market_name: str) -> dict:
    """Get current and recent market price trend for a crop at a market.

    Args:
        crop: Crop name.
        market_name: Local market/mandi name.

    Returns:
        dict with price_per_kg, currency, trend_pct_7d.
    """
    # STUB: replace with a real market-price API call. Numbers below are
    # illustrative only.
    seed = sum(ord(c) for c in crop + market_name)
    rnd = random.Random(seed)
    price = round(rnd.uniform(8, 40), 2)
    trend = round(rnd.uniform(-15, 15), 1)
    return {
        "crop": crop,
        "market": market_name,
        "price_per_kg": price,
        "currency": "local_unit",
        "trend_pct_7d": trend,
        "as_of": datetime.now().strftime("%Y-%m-%d"),
        "source": "STUB DATA — replace with real market price API/MCP server",
    }


# ---------------------------------------------------------------------------
# 4. ECONOMIC DECISION — REAL LOGIC. Compares cost of treating the crop
#    against the expected value of yield saved. This is the core novel
#    piece: it's the calculation existing diagnosis-only or price-only
#    apps don't do.
# ---------------------------------------------------------------------------
def compute_economics(
    treatment_cost_per_hectare: float,
    expected_yield_loss_pct_if_untreated: float,
    expected_yield_kg_per_hectare: float,
    price_per_kg: float,
    days_to_harvest: int,
) -> dict:
    """Decide whether treating the crop is economically worthwhile.

    Args:
        treatment_cost_per_hectare: Cost of the fungicide/pesticide + labor.
        expected_yield_loss_pct_if_untreated: Estimated % yield lost if the
            disease/pest is left untreated (agronomist estimate or lookup
            table by disease severity).
        expected_yield_kg_per_hectare: Expected total yield per hectare.
        price_per_kg: Current market price per kg.
        days_to_harvest: Days remaining until harvest.

    Returns:
        dict with recommendation, expected_loss_value, net_benefit.
    """
    expected_loss_kg = expected_yield_kg_per_hectare * (
        expected_yield_loss_pct_if_untreated / 100
    )
    expected_loss_value = round(expected_loss_kg * price_per_kg, 2)
    net_benefit = round(expected_loss_value - treatment_cost_per_hectare, 2)

    if days_to_harvest <= 7:
        # Too close to harvest — treatment may not have time to help, and
        # pre-harvest interval (PHI) restrictions likely apply.
        recommendation = "do_not_spray_too_close_to_harvest"
        reason = (
            f"Only {days_to_harvest} days to harvest — check the pre-harvest "
            "interval on the product label before spraying."
        )
    elif net_benefit > treatment_cost_per_hectare * 0.25:
        recommendation = "spray"
        reason = (
            f"Expected loss value ({expected_loss_value}) clearly exceeds "
            f"treatment cost ({treatment_cost_per_hectare}). Net benefit: "
            f"{net_benefit}."
        )
    elif net_benefit > 0:
        recommendation = "spray_marginal"
        reason = (
            "Treating is worthwhile but the margin is thin "
            f"(net benefit: {net_benefit}). Consider a cheaper input option."
        )
    else:
        recommendation = "do_not_spray"
        reason = (
            f"Treatment cost ({treatment_cost_per_hectare}) exceeds expected "
            f"loss value ({expected_loss_value}). Not economically justified."
        )

    return {
        "recommendation": recommendation,
        "reason": reason,
        "expected_loss_value": expected_loss_value,
        "treatment_cost": treatment_cost_per_hectare,
        "net_benefit": net_benefit,
    }


# ---------------------------------------------------------------------------
# 5. ACTIONS — draft is safe/no side effects. send_dealer_order and
#    schedule_reminder actually do something and must require confirmation.
#    The require_confirmation=True flag is set in agents.py when these are
#    registered as tools, not here — this file only defines behavior.
# ---------------------------------------------------------------------------
def draft_dealer_order(crop: str, product_name: str, quantity_kg: float, farmer_name: str) -> dict:
    """Draft (but do not send) a purchase order message to a local input dealer."""
    message = (
        f"Order request from {farmer_name}: {quantity_kg}kg of {product_name} "
        f"for {crop} treatment. Please confirm availability and price."
    )
    return {"draft_message": message, "status": "drafted_not_sent"}


def send_dealer_order(dealer_contact: str, order_message: str) -> dict:
    """Actually send the order message to the dealer (e.g. via WhatsApp Business API).

    This is a side-effecting action and MUST require human confirmation
    before it runs (enforced via require_confirmation=True in agents.py).
    """
    # STUB: replace with a real WhatsApp Business Cloud API / Twilio call.
    return {
        "status": "sent (SIMULATED — no real message was sent)",
        "dealer_contact": dealer_contact,
        "message": order_message,
        "sent_at": datetime.now().isoformat(),
    }


def schedule_reapplication_reminder(crop: str, days_from_now: int) -> dict:
    """Schedule a reminder to re-check/reapply treatment."""
    reminder_date = (datetime.now() + timedelta(days=days_from_now)).strftime("%Y-%m-%d")
    return {
        "crop": crop,
        "reminder_date": reminder_date,
        "status": "scheduled (SIMULATED — hook up to a real calendar/SMS reminder service)",
    }


Overwriting tools.py


In [112]:
# Quick sanity check of the real logic (no API key needed for this part)
from tools import get_weather_forecast, compute_disease_risk, get_market_price, compute_economics

w = get_weather_forecast("Jimma")
r = compute_disease_risk(w["avg_temp_c"], w["avg_humidity_pct"], "tomato")
p = get_market_price("tomato", "Jimma Market")
e = compute_economics(
    treatment_cost_per_hectare=1800,
    expected_yield_loss_pct_if_untreated=25,
    expected_yield_kg_per_hectare=20000,
    price_per_kg=p["price_per_kg"],
    days_to_harvest=20,
)
print("Weather:", w)
print("Disease risk:", r)
print("Market price:", p)
print("Economic decision:", e)


Weather: {'location': 'Jimma', 'forecast_days': 5, 'avg_temp_c': 29.5, 'avg_humidity_pct': 76.0, 'rain_expected_mm': 25.5, 'source': 'STUB DATA — replace with real weather API'}
Disease risk: {'crop': 'tomato', 'risk_score': 40, 'risk_level': 'moderate', 'rationale': 'At 76.0% humidity and 29.5C, conditions are favorable for fungal disease spread on tomato.'}
Market price: {'crop': 'tomato', 'market': 'Jimma Market', 'price_per_kg': 23.36, 'currency': 'local_unit', 'trend_pct_7d': 3.2, 'as_of': '2026-07-06', 'source': 'STUB DATA — replace with real market price API/MCP server'}
Economic decision: {'recommendation': 'spray', 'reason': 'Expected loss value (116800.0) clearly exceeds treatment cost (1800). Net benefit: 115000.0.', 'expected_loss_value': 116800.0, 'treatment_cost': 1800, 'net_benefit': 115000.0}


## 3. Agents

Five specialist agents behind one orchestrator, built on the real
`google-adk` API (`Agent` + `sub_agents` delegation).


In [113]:
%%writefile agents.py
"""
agents.py — The AgriMargin multi-agent system, built on real google-adk
(pip install google-adk, version 2.x — verified against the installed
package, not guessed).

Pattern used: a root "orchestrator" Agent with sub_agents. ADK's LlmAgent
automatically exposes each sub-agent as something the parent can delegate
to mid-conversation (transfer_to_agent), so the orchestrator can reason
about a farmer's message, call the right specialists in order, and
combine their outputs — without you hand-writing the control flow.

Model: all agents use "gemini-2.5-flash" by default — fast and cheap
enough for a conversational agronomy assistant. Swap to a stronger model
for the Diagnosis Agent if photo-ID accuracy needs improving.
"""

import os
from google.adk import Agent
from google.adk.tools import FunctionTool

from tools import (
    get_weather_forecast,
    compute_disease_risk,
    get_market_price,
    compute_economics,
    draft_dealer_order,
    send_dealer_order,
    schedule_reapplication_reminder,
)

MODEL = os.environ.get("AGRIMARGIN_MODEL", "gemini-2.5-flash")


# ---------------------------------------------------------------------------
# 1. Diagnosis Agent
# ---------------------------------------------------------------------------
# In production this agent receives the farmer's photo directly as a
# multimodal message part (Gemini's native vision, not a function tool) —
# ADK passes images straight to the model. For text-only testing (no
# photo available), it reasons from a symptom description instead.
diagnosis_agent = Agent(
    name="diagnosis_agent",
    model=MODEL,
    description=(
        "Identifies crop pest/disease from a photo or a text description of "
        "symptoms, and reports a confidence level."
    ),
    instruction=(
        "You are a plant pathology expert. You will be given either an image "
        "of a crop leaf/plant, or a text description of symptoms. Identify "
        "the most likely pest or disease. Respond with: the crop, the "
        "diagnosis, your confidence (low/medium/high), and one sentence of "
        "reasoning. If the input is too ambiguous to diagnose confidently, "
        "say so plainly instead of guessing — do not invent a diagnosis you "
        "are not reasonably confident in."
    ),
)


# ---------------------------------------------------------------------------
# 2. Weather-Risk Agent
# ---------------------------------------------------------------------------
weather_risk_agent = Agent(
    name="weather_risk_agent",
    model=MODEL,
    description="Fetches forecast weather and computes disease-spread risk for a crop.",
    instruction=(
        "You assess disease-spread risk from weather. Given a location and "
        "crop, call get_weather_forecast to get the forecast, then call "
        "compute_disease_risk with those numbers. Report the risk_level and "
        "rationale clearly. Do not skip calling the tools — do not estimate "
        "weather from general knowledge."
    ),
    tools=[
        FunctionTool(get_weather_forecast),
        FunctionTool(compute_disease_risk),
    ],
)


# ---------------------------------------------------------------------------
# 3. Market-Intelligence Agent
# ---------------------------------------------------------------------------
# NOTE: get_market_price is a local stub function right now. To satisfy the
# capstone's "MCP server" requirement for real, point this agent at an
# MCPToolset instead — see README.md "Swapping in a real MCP server".
market_intel_agent = Agent(
    name="market_intel_agent",
    model=MODEL,
    description="Fetches current market/mandi price and short-term trend for a crop.",
    instruction=(
        "You report crop market prices. Given a crop and market name, call "
        "get_market_price and report the price, currency, and 7-day trend. "
        "Always call the tool — never guess a price from memory."
    ),
    tools=[FunctionTool(get_market_price)],
)


# ---------------------------------------------------------------------------
# 4. Economic-Advisor Agent
# ---------------------------------------------------------------------------
economic_advisor_agent = Agent(
    name="economic_advisor_agent",
    model=MODEL,
    description=(
        "Combines diagnosis, disease risk, and market price into a spray / "
        "wait / sell recommendation with the underlying cost-benefit math."
    ),
    instruction=(
        "You are the decision-making core of this system. You will be given "
        "a diagnosis, a disease risk level, and a market price, plus farm "
        "figures (treatment cost, expected yield, days to harvest). Call "
        "compute_economics with your best estimates for "
        "expected_yield_loss_pct_if_untreated based on the diagnosis "
        "confidence and risk level (low risk+low confidence diagnosis = "
        "lower estimate, e.g. 5-10%; high risk+high confidence = higher, "
        "e.g. 20-35%). State your recommendation plainly, show the numbers, "
        "and explain the reasoning in one paragraph a farmer can actually "
        "understand — no jargon. If any critical figure is missing (crop "
        "price, yield estimate, treatment cost), ask for it instead of "
        "guessing."
    ),
    tools=[FunctionTool(compute_economics)],
)


# ---------------------------------------------------------------------------
# 5. Action Agent — the only agent allowed to touch anything external.
# ---------------------------------------------------------------------------
# send_dealer_order is registered with require_confirmation=True. ADK will
# pause and require an explicit human confirmation before this tool
# actually executes — this is the safety rail. draft_dealer_order has no
# side effects, so it does not need confirmation.
action_agent = Agent(
    name="action_agent",
    model=MODEL,
    description=(
        "Drafts and (with explicit farmer confirmation) sends a dealer "
        "order, and schedules a reapplication reminder. Never acts without "
        "confirmation."
    ),
    instruction=(
        "You take action ONLY after the economic_advisor_agent has "
        "recommended spraying. First call draft_dealer_order to prepare the "
        "order message and show it to the farmer. Only call send_dealer_order "
        "after the farmer has explicitly approved the draft — never send "
        "without approval, and never treat silence as approval. After a "
        "successful send, call schedule_reapplication_reminder for 10-14 "
        "days out. If the economic recommendation was not to spray, do not "
        "draft or send any order — just confirm to the farmer that no "
        "action is needed."
    ),
    tools=[
        FunctionTool(draft_dealer_order),
        FunctionTool(send_dealer_order, require_confirmation=True),
        FunctionTool(schedule_reapplication_reminder, require_confirmation=True),
    ],
)


# ---------------------------------------------------------------------------
# Orchestrator — root agent, delegates to the five specialists above.
# ---------------------------------------------------------------------------
orchestrator = Agent(
    name="agrimargin_orchestrator",
    model=MODEL,
    description=(
        "AgriMargin: helps a farmer decide whether to spray, wait, or sell, "
        "and can act on that decision."
    ),
    instruction=(
        "You are AgriMargin, an assistant that helps a smallholder farmer "
        "make an economically sound decision about a crop problem, then "
        "optionally acts on it. For a new crop-problem query, work through "
        "these steps in order, delegating to the right specialist agent for "
        "each: \n"
        "1. diagnosis_agent — identify the pest/disease.\n"
        "2. weather_risk_agent — assess disease-spread risk for the farmer's "
        "location and crop.\n"
        "3. market_intel_agent — get the current price for the crop at the "
        "farmer's local market.\n"
        "4. economic_advisor_agent — combine all of the above into a spray/"
        "wait/sell recommendation, asking the farmer for any missing farm "
        "figures (treatment cost, expected yield per hectare, days to "
        "harvest) if needed.\n"
        "5. Only if the recommendation is to spray AND the farmer wants "
        "help acting on it, delegate to action_agent.\n"
        "Keep your own messages to the farmer short, plain-language, and "
        "in the language they're writing in. Never skip the economic step "
        "and jump straight to 'yes, spray' just because a disease was "
        "identified — the whole point of this system is that diagnosis "
        "alone is not a recommendation."
    ),
    sub_agents=[
        diagnosis_agent,
        weather_risk_agent,
        market_intel_agent,
        economic_advisor_agent,
        action_agent,
    ],
)


root_agent = orchestrator  # ADK convention: root_agent is the entrypoint.


Overwriting agents.py


In [114]:
from agents import orchestrator, root_agent

print("Orchestrator:", orchestrator.name)
print("Sub-agents:", [a.name for a in orchestrator.sub_agents])
for a in orchestrator.sub_agents:
    print(" ", a.name, "-> tools:", [t.name if hasattr(t, "name") else t.__name__ for t in a.tools])


Orchestrator: agrimargin_orchestrator
Sub-agents: ['diagnosis_agent', 'weather_risk_agent', 'market_intel_agent', 'economic_advisor_agent', 'action_agent']
  diagnosis_agent -> tools: []
  weather_risk_agent -> tools: ['get_weather_forecast', 'compute_disease_risk']
  market_intel_agent -> tools: ['get_market_price']
  economic_advisor_agent -> tools: ['compute_economics']
  action_agent -> tools: ['draft_dealer_order', 'send_dealer_order', 'schedule_reapplication_reminder']


## 4. Run AgriMargin

In [115]:
import asyncio
from google.adk import Runner
from google.adk.sessions import InMemorySessionService
from agents import root_agent

APP_NAME, USER_ID, SESSION_ID = "agrimargin", "demo_farmer", "demo_session"

DEMO_MESSAGE = (
    "My tomato leaves have dark concentric rings on the lower leaves and "
    "some yellowing around them. Weather has been humid, I'm near Jimma. "
    "Should I spray? Treatment costs about 1800 per hectare, I expect "
    "about 20000kg/hectare yield, and harvest is 20 days away. Market is "
    "the Jimma Market."
)

async def run_demo():
    session_service = InMemorySessionService()
    await session_service.create_session(app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID)
    runner = Runner(agent=root_agent, app_name=APP_NAME, session_service=session_service)
    runner.run_debug(DEMO_MESSAGE, user_id=USER_ID, session_id=SESSION_ID)

await run_demo()  # Colab supports top-level await

/tmp/ipykernel_13412/103748343.py:20: RuntimeWarning: coroutine 'Runner.run_debug' was never awaited
  runner.run_debug(DEMO_MESSAGE, user_id=USER_ID, session_id=SESSION_ID)


In [116]:
# Optional: ask your own question
USER_MESSAGE = "Is it worth spraying my maize for fall armyworm with 10 days to harvest?"

async def run_custom():
    session_service = InMemorySessionService()
    await session_service.create_session(app_name=APP_NAME, user_id=USER_ID, session_id="custom_session")
    runner = Runner(agent=root_agent, app_name=APP_NAME, session_service=session_service)
    runner.run_debug(USER_MESSAGE, user_id=USER_ID, session_id="custom_session")

await run_custom()

/tmp/ipykernel_13412/3564041740.py:8: RuntimeWarning: coroutine 'Runner.run_debug' was never awaited
  runner.run_debug(USER_MESSAGE, user_id=USER_ID, session_id="custom_session")


## 5. Architecture

```
agrimargin_orchestrator (root Agent)
 ├── diagnosis_agent        — pest/disease ID from photo or symptoms
 ├── weather_risk_agent     — forecast → disease-spread risk index
 ├── market_intel_agent     — current crop price + trend
 ├── economic_advisor_agent — spray / wait / sell recommendation + math
 └── action_agent           — drafts + (confirmed) sends dealer order,
                               schedules reapplication reminder
```

The orchestrator is a standard ADK `Agent` with `sub_agents=[...]` - it
delegates to each specialist automatically based on the conversation
(ADK's built-in `transfer_to_agent`), rather than a hand-coded pipeline.

**Security feature:** `send_dealer_order` and
`schedule_reapplication_reminder` are registered with
`require_confirmation=True` in `agents.py` - ADK will not execute either
without an explicit human "yes". The agent cannot place an order or send
a message unattended.
